In [ ]:
# 1
!pip install scikit-learn scipy pandas numpy tqdm joblib implicit -q
import warnings
warnings.filterwarnings('ignore')
print('Cài đặt hoàn tất')

In [ ]:
#2
import os, glob, gc, pickle, json, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from tqdm import tqdm
from collections import defaultdict, deque
from implicit.als import AlternatingLeastSquares
import implicit.cpu.als as cpu_als
import joblib

CONFIG = {
    'data_dir'        : '/kaggle/input/datasets/js042710/second3t1k',
    'output_dir'      : '/kaggle/working/model',
    'checkpoint_dir'  : '/kaggle/working/checkpoints',
    'factors'         : 256,
    'iterations'      : 30,
    'regularization'  : 0.05,
    'alpha'           : 15,
    'use_gpu'         : True,
    'num_threads'     : 4,
    'min_interactions': 20,
    'item_key'        : 'track_artist',
    'chunk_size'      : 500_000,
}

try:
    import implicit.gpu
    HAS_GPU = implicit.gpu.HAS_CUDA
except:
    HAS_GPU = False

CONFIG['use_gpu'] = HAS_GPU
print(f'GPU available: {HAS_GPU}')
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
print('Config loaded')

In [ ]:
#3
all_files = sorted(glob.glob(os.path.join(CONFIG['data_dir'], '**', '*.csv'), recursive=True))
all_files += sorted(glob.glob(os.path.join(CONFIG['data_dir'], '**', '*.xlsx'), recursive=True))

print(f'Tổng số file: {len(all_files)}')
if len(all_files) > 0:
    df_sample = pd.read_csv(all_files[0], nrows=5)
    display(df_sample.head(2))
else:
    print("❌ Không tìm thấy file nào!")

In [ ]:
#4
class InteractionAccumulator:
    def __init__(self):
        self.user2idx     = {}
        self.item2idx     = {}
        self.idx2user     = []
        self.idx2item     = []
        self.interactions = defaultdict(float)
        self.temporal_history = defaultdict(lambda: deque(maxlen=200)) 
        self.n_rows_processed = 0

    def _get_or_create(self, mapping, reverse, key):
        if key not in mapping:
            mapping[key] = len(reverse)
            reverse.append(key)
        return mapping[key]

    def add_dataframe(self, df: pd.DataFrame):
        cols = ['user_id','track_name','artist_name', 'timestamp']
        df = df[cols].dropna()
        df['item_key'] = df['track_name'].str.strip() + '|||' + df['artist_name'].str.strip()
        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
        df = df.dropna(subset=['timestamp'])

        for row in df.itertuples(index=False):
            uid = self._get_or_create(self.user2idx, self.idx2user, str(row.user_id))
            iid = self._get_or_create(self.item2idx, self.idx2item, row.item_key)
            
            self.interactions[(uid, iid)] += 1.0 
            self.temporal_history[uid].append((iid, int(row.timestamp.timestamp())))
            
        self.n_rows_processed += len(df)

    def build_matrix(self, min_interactions=20):
        print(f' Building matrix từ {len(self.interactions):,} cặp tương tác...')
        rows, cols, data = [], [], []
        for (uid, iid), cnt in self.interactions.items():
            rows.append(uid); cols.append(iid); data.append(cnt)
        mat = sp.csr_matrix(
            (data,(rows,cols)),
            shape=(len(self.idx2user), len(self.idx2item)),
            dtype=np.float32
        )
        if min_interactions > 1:
            mat_bin = (mat > 0).astype(np.float32)
            ic_cnt  = np.asarray(mat_bin.sum(axis=0)).flatten()
            uc_cnt  = np.asarray(mat_bin.sum(axis=1)).flatten()
            vi = ic_cnt >= min_interactions
            vu = uc_cnt >= min_interactions
            
            # --- TẠO MAPPING CHUYỂN TỪ ID CŨ SANG ID MỚI ---
            old_item2new = {old: new for new, old in enumerate(np.where(vi)[0])}
            old_user2new = {old: new for new, old in enumerate(np.where(vu)[0])}
            
            mat = mat[vu][:, vi]
            self.idx2user = [u for u,v in zip(self.idx2user, vu) if v]
            self.idx2item = [i for i,v in zip(self.idx2item, vi) if v]
            self.user2idx = {u:i for i,u in enumerate(self.idx2user)}
            self.item2idx = {i:j for j,i in enumerate(self.idx2item)}
            
            # --- CẬP NHẬT LẠI ID TRONG TEMPORAL HISTORY ---
            new_history = defaultdict(lambda: deque(maxlen=200))
            for old_u, hist in self.temporal_history.items():
                if old_u in old_user2new:
                    new_u = old_user2new[old_u]
                    for old_i, ts in hist:
                        if old_i in old_item2new:
                            new_history[new_u].append((old_item2new[old_i], ts))
            self.temporal_history = new_history
            
        if mat.shape[0] > 0 and mat.shape[1] > 0:
            density = mat.nnz / (mat.shape[0] * mat.shape[1]) * 100
        else:
            density = 0.0
            
        print(f'   Ma trận: {mat.shape[0]:,} × {mat.shape[1]:,} | Density: {density:.4f}%')
        return mat

print('Khởi tạo InteractionAccumulator hoàn tất')

In [ ]:
#5
CHECKPOINT_INDEX_PATH = os.path.join(CONFIG['checkpoint_dir'], 'accumulator_checkpoint.pkl')
PROGRESS_PATH = os.path.join(CONFIG['checkpoint_dir'], 'progress.json')

def save_accumulator_checkpoint(acc, processed_files):
    state = {
        'user2idx': acc.user2idx, 'item2idx': acc.item2idx,
        'idx2user': acc.idx2user, 'idx2item': acc.idx2item,
        'interactions': dict(acc.interactions),
        'temporal_history': {k: list(v) for k, v in acc.temporal_history.items()},
        'n_rows_processed': acc.n_rows_processed,
    }
    joblib.dump(state, CHECKPOINT_INDEX_PATH, compress=3)
    with open(PROGRESS_PATH, 'w') as f:
        json.dump({'processed_files': processed_files}, f)

def load_accumulator_checkpoint():
    if not os.path.exists(CHECKPOINT_INDEX_PATH):
        return InteractionAccumulator(), []
    state = joblib.load(CHECKPOINT_INDEX_PATH)
    acc = InteractionAccumulator()
    acc.user2idx, acc.item2idx = state['user2idx'], state['item2idx']
    acc.idx2user, acc.idx2item = state['idx2user'], state['idx2item']
    acc.interactions = defaultdict(float, state['interactions'])
    acc.temporal_history = defaultdict(lambda: deque(maxlen=200), {k: deque(v, maxlen=200) for k, v in state.get('temporal_history', {}).items()})
    acc.n_rows_processed = state['n_rows_processed']
    with open(PROGRESS_PATH) as f:
        progress = json.load(f)
    return acc, progress['processed_files']

RESUME_FROM_CHECKPOINT = False # Chạy lần đầu nên để False
if RESUME_FROM_CHECKPOINT:
    accumulator, processed_files = load_accumulator_checkpoint()
else:
    accumulator, processed_files = InteractionAccumulator(), []

remaining_files = [f for f in all_files if f not in processed_files]
print(f'Còn {len(remaining_files)} files cần xử lý')

In [ ]:
#6
SAVE_EVERY_N_FILES = 15   

t0 = time.time()
for i, fpath in enumerate(tqdm(remaining_files, desc='Loading files')):
    fname = os.path.basename(fpath)
    try:
        if fpath.endswith('.xlsx'):
            df = pd.read_excel(fpath, usecols=['user_id','track_name','artist_name', 'timestamp'])
            accumulator.add_dataframe(df)
        else:
            for chunk in pd.read_csv(
                fpath,
                usecols=['user_id','track_name','artist_name', 'timestamp'],
                chunksize=CONFIG['chunk_size'],
                low_memory=False
            ):
                accumulator.add_dataframe(chunk)
                del chunk
                gc.collect()

        processed_files.append(fpath)
        if (i + 1) % SAVE_EVERY_N_FILES == 0:
            save_accumulator_checkpoint(accumulator, processed_files)

    except Exception as e:
        print(f'Lỗi file {fname}: {e}')
        continue

save_accumulator_checkpoint(accumulator, processed_files)
print(f'\nXử lý xong {len(processed_files)} files trong {(time.time() - t0)/60:.1f} phút')

In [ ]:
#7
user_item_matrix = accumulator.build_matrix(min_interactions=CONFIG['min_interactions'])
item_user_matrix = user_item_matrix.T.tocsr()

def user_based_split(user_item, test_ratio=0.2, min_items=5):
    train = user_item.tolil().astype(np.float32)
    test  = sp.lil_matrix(user_item.shape, dtype=np.float32)
    rng   = np.random.default_rng(42)
    for u in range(user_item.shape[0]):
        items = user_item[u].indices
        if len(items) < min_items: continue
        n_test = max(1, int(len(items) * test_ratio))
        test_items = rng.choice(items, n_test, replace=False)
        for it in test_items:
            test[u, it]  = user_item[u, it]
            train[u, it] = 0
    return train.tocsr(), test.tocsr()

print('Đang tạo user-based split...')
train_matrix, test_matrix = user_based_split(user_item_matrix)
train_item_user = train_matrix.T.tocsr()
print(f'Train: {train_matrix.nnz:,} | Test: {test_matrix.nnz:,}')

In [ ]:
#8
print(f'Train ALS trên {"GPU" if CONFIG["use_gpu"] else "CPU"}...')
model = AlternatingLeastSquares(
    factors=CONFIG['factors'], iterations=CONFIG['iterations'],
    regularization=CONFIG['regularization'], alpha=CONFIG['alpha'],
    use_gpu=CONFIG['use_gpu'], num_threads=CONFIG.get('num_threads', 4),
    calculate_training_loss=True, random_state=42
)
t0 = time.time()
model.fit(train_item_user)
print(f'Train xong trong {(time.time()-t0)/60:.1f} phút')

if CONFIG['use_gpu']:
    print('Converting GPU → CPU...')
    item_f, user_f = model.item_factors.to_numpy(), model.user_factors.to_numpy()
    cpu_model = cpu_als.AlternatingLeastSquares(factors=CONFIG['factors'], iterations=0, regularization=CONFIG['regularization'])
    cpu_model.item_factors, cpu_model.user_factors = user_f, item_f   
    model = cpu_model
    print('Convert xong')

In [ ]:
# 9
def evaluate_model(model, train_mat, test_mat, K=10, n_users=2000):
    rng = np.random.default_rng(42)
    n_eval = min(n_users, test_mat.shape[0])
    test_users = rng.choice(test_mat.shape[0], n_eval, replace=False)
    ap_list, prec_list, rec_list = [], [], []

    for u in test_users:
        actual = set(test_mat[u].indices)
        if not actual: continue
        rec_ids, _ = model.recommend(u, train_mat[u], N=K, filter_already_liked_items=True)
        rec_ids = list(rec_ids)

        hits = len(set(rec_ids) & actual)
        prec_list.append(hits / K)
        rec_list.append(hits / len(actual))

        ap, n_hits = 0.0, 0
        for rank, item in enumerate(rec_ids, 1):
            if item in actual:
                n_hits += 1
                ap += n_hits / rank
        ap_list.append(ap / min(len(actual), K))

    print(f'Kết quả (n_users={n_eval}, K={K}): MAP: {np.mean(ap_list):.4f} | Precision: {np.mean(prec_list):.4f}')
    return np.mean(ap_list), np.mean(prec_list), np.mean(rec_list)

map_at_10, p10, r10 = evaluate_model(model, train_matrix, test_matrix, K=10)

In [ ]:
#10
FINAL_MODEL_DIR = CONFIG['output_dir']
joblib.dump(model, os.path.join(FINAL_MODEL_DIR, 'als_model.pkl'), compress=3)

joblib.dump({
    'user2idx': accumulator.user2idx, 'item2idx': accumulator.item2idx,
    'idx2user': accumulator.idx2user, 'idx2item': accumulator.idx2item,
    'temporal_history': {k: list(v) for k, v in accumulator.temporal_history.items()}, 
}, os.path.join(FINAL_MODEL_DIR, 'index_mappings.pkl'), compress=3)

sp.save_npz(os.path.join(FINAL_MODEL_DIR, 'user_item_matrix.npz'), user_item_matrix)

item_meta = pd.DataFrame([{'item_idx': i, 'track_name': k.split('|||')[0], 'artist_name': k.split('|||')[1] if '|||' in k else 'Unknown'} for i, k in enumerate(accumulator.idx2item)])
item_meta.to_parquet(os.path.join(FINAL_MODEL_DIR, 'item_metadata.parquet'), index=False)
print('Đã lưu xong toàn bộ Model và Mappings!')

In [ ]:
#11
class MusicRecommender:
    def __init__(self, model_dir: str):
        self.model     = joblib.load(os.path.join(model_dir, 'als_model.pkl'))
        mappings       = joblib.load(os.path.join(model_dir, 'index_mappings.pkl'))
        self.user2idx, self.item2idx = mappings['user2idx'], mappings['item2idx']
        self.idx2user, self.idx2item = mappings['idx2user'], mappings['idx2item']
        self.temporal_history = mappings.get('temporal_history', {}) 
        self.user_item = sp.load_npz(os.path.join(model_dir, 'user_item_matrix.npz'))

    def _build_temporal_user_vector(self, user_idx, target_date):
        target_ts = pd.to_datetime(target_date).timestamp()
        history = self.temporal_history.get(user_idx, [])
        past_interactions = [item for item in history if item[1] <= target_ts]
        if not past_interactions: return None

        counts = defaultdict(float)
        for iid, ts in past_interactions:
            days_diff = (target_ts - ts) / 86400
            weight = np.exp(-max(0, days_diff) / 14.0) 
            counts[iid] += weight

        cols, data = list(counts.keys()), list(counts.values())
        return sp.csr_matrix((data, ([0]*len(cols), cols)), shape=(1, len(self.idx2item)), dtype=np.float32)

    def recommend_by_timeline(self, user_id, target_date, n=10):
        uid_str = str(user_id)
        if uid_str not in self.user2idx: return self.popular_items(n)
        uid = self.user2idx[uid_str]
        
        user_vector = self._build_temporal_user_vector(uid, target_date)
        if user_vector is None: return self.popular_items(n)

        rec_ids, scores = self.model.recommend(userid=0, user_items=user_vector, N=n, filter_already_liked_items=True, recalculate_user=True)
        return self._item_idx_to_df(list(rec_ids), list(scores))

    def _item_idx_to_df(self, item_ids, scores):
        rows = []
        for idx, score in zip(item_ids, scores):
            key = self.idx2item[idx]
            parts = key.split('|||')
            rows.append({'track_name': parts[0] if len(parts)>0 else "Unknown", 'artist_name': parts[1] if len(parts)>1 else "Unknown", 'score': round(float(score),4)})
        return pd.DataFrame(rows)

    def user_history(self, user_id, n=20):
        uid_str = str(user_id)
        if uid_str not in self.user2idx: return pd.DataFrame()
        row = self.user_item[self.user2idx[uid_str]]
        top_n = np.argsort(row.data)[::-1][:n]
        df = self._item_idx_to_df(row.indices[top_n], row.data[top_n])
        df.rename(columns={'score':'weighted_plays'}, inplace=True)
        df.index = range(1, len(df)+1)
        return df

    def popular_items(self, n=10):
        item_counts = np.asarray(self.user_item.sum(axis=0)).flatten()
        top_ids = np.argsort(item_counts)[::-1][:n]
        df = self._item_idx_to_df(top_ids, item_counts[top_ids])
        df.rename(columns={'score':'play_count'}, inplace=True)
        df.index = range(1, len(df)+1)
        return df

    def similar_users(self, user_id, n=5):
        uid_str = str(user_id)
        if uid_str not in self.user2idx: return pd.DataFrame()
        uid = self.user2idx[uid_str]
        sim_ids, scores = self.model.similar_users(uid, N=n+1)
        filtered = [(i,s) for i,s in zip(list(sim_ids),list(scores)) if i != uid][:n]
        if not filtered: return pd.DataFrame()
        ids, sims = zip(*filtered)
        df = pd.DataFrame({'user_id': [self.idx2user[i] for i in ids], 'similarity': [round(float(s),4) for s in sims]})
        df.index = range(1, len(df)+1)
        return df

print('MusicRecommender sẵn sàng')

In [ ]:
#12
rec = MusicRecommender(CONFIG['output_dir'])
TEST_USER_ID = 1 
u_idx = rec.user2idx.get(str(TEST_USER_ID), -1)
print(f'User {TEST_USER_ID} có {len(rec.temporal_history.get(u_idx, []))} bản ghi lịch sử.')
display(rec.user_history(TEST_USER_ID, n=5))

In [ ]:
#13
DATE_1 = "2026-01-20"
print(f"=== [SLIDER TẠI NGÀY: {DATE_1}] ===")
res_1 = rec.recommend_by_timeline(TEST_USER_ID, DATE_1, n=10)
display(res_1)

DATE_2 = "2026-03-01" 
print(f"\n=== [SLIDER TẠI NGÀY: {DATE_2}] ===")
res_2 = rec.recommend_by_timeline(TEST_USER_ID, DATE_2, n=10)
display(res_2)

diff = set(res_1['track_name']) != set(res_2['track_name'])
print(f"\n=> Gợi ý có thay đổi theo thời gian không? {'CÓ (THÀNH CÔNG)' if diff else 'KHÔNG'}")

In [ ]:
#14
print(f'--- USERS CÓ GU NHẠC GIỐNG USER {TEST_USER_ID} ---')
display(rec.similar_users(TEST_USER_ID, n=5))

print('\n--- TOP BÀI HÁT PHỔ BIẾN NHẤT (Cold Start) ---')
display(rec.popular_items(n=10))

In [ ]:
#15
def update_model_with_new_data(new_file_paths: list, model_dir: str, n_iter_update=10):
    old_acc, old_processed = load_accumulator_checkpoint()
    for fpath in tqdm(new_file_paths, desc='Loading new data'):
        for chunk in pd.read_csv(fpath, usecols=['user_id','track_name','artist_name', 'timestamp'], chunksize=500_000):
            old_acc.add_dataframe(chunk)
    new_matrix = old_acc.build_matrix(min_interactions=CONFIG['min_interactions'])
    new_model = AlternatingLeastSquares(factors=CONFIG['factors'], iterations=n_iter_update, regularization=CONFIG['regularization'], alpha=CONFIG['alpha'], use_gpu=CONFIG['use_gpu'], random_state=42)
    new_model.fit(new_matrix.T.tocsr())
    joblib.dump(new_model, os.path.join(model_dir, 'als_model.pkl'), compress=3)
    joblib.dump({'user2idx': old_acc.user2idx, 'item2idx': old_acc.item2idx, 'idx2user': old_acc.idx2user, 'idx2item': old_acc.idx2item, 'temporal_history': {k: list(v) for k, v in old_acc.temporal_history.items()}}, os.path.join(model_dir, 'index_mappings.pkl'), compress=3)
    sp.save_npz(os.path.join(model_dir, 'user_item_matrix.npz'), new_matrix)
    save_accumulator_checkpoint(old_acc, old_processed + new_file_paths)
    print('Update hoàn tất!')